In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import re
import nltk

In [2]:
nltk_resources = [
    'tokenizers/punkt', 
    'averaged_perceptron_tagger_eng',
    'corpora/stopwords', 
    'help/tagsets'
]

for resource in nltk_resources:
    try:
        nltk.data.find(resource)
    except IndexError:
        nltk.download(resource)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [3]:
data_home = "../input"
output_dir = "../working"

source_file_dir = '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels'
source_file_dir

'/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels'

In [4]:
# This structure matches all the texts
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

# Gutenberg boilerplate clip patterns (common to all texts)
clip_pats_gutenberg = [
    r"\*\*\*\s*START OF",
    r"\*\*\*\s*END OF"
]

# Roman numeral character class for reuse
roman = '[IVXLCM]+'

# Chapter heading patterns per book (book_id, regex)
# 696   = The Castle of Otranto (Walpole) — "CHAPTER I." style, all caps, period
# 3268  = The Mysteries of Udolpho (Radcliffe) — two-level: VOLUME N then " CHAPTER I" (one leading space)
# 5182  = The Old English Baron (Reeve) — no chapter divisions; treat as single chapter
chap_pats = [
    (696,  rf"^CHAPTER\s+{roman}\.$"),
    (3268, rf"^(?:VOLUME\s+\d+|\ CHAPTER\s+{roman})$"),
    (5182, rf"^THE OLD ENGLISH BARON: A GOTHIC STORY\.$"),  # single-chapter fallback
]

In [5]:
source_file_list = sorted(glob(f"{source_file_dir}/*.*"))
source_file_list

['/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/RADCLIFFE_ANN_THE_MYSTERIES_OF_UDOLPHO-pg3268.txt',
 '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/REEVE_CLARA_THE_OLD_ENGLISH_BARON-pg5182.txt',
 '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/WALPOLE_HORACE_CASTLE_OF_OTRANTO-pg696.txt']

In [6]:
book_data = []
for source_file_path in source_file_list:
    book_id = int(source_file_path.split('-')[-1].split('.')[0].replace('pg',''))
    raw_title = source_file_path.split('/')[-1].split('-')[0].replace('_', ' ')
    book_data.append((book_id, source_file_path, raw_title))

book_data

[(3268,
  '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/RADCLIFFE_ANN_THE_MYSTERIES_OF_UDOLPHO-pg3268.txt',
  'RADCLIFFE ANN THE MYSTERIES OF UDOLPHO'),
 (5182,
  '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/REEVE_CLARA_THE_OLD_ENGLISH_BARON-pg5182.txt',
  'REEVE CLARA THE OLD ENGLISH BARON'),
 (696,
  '/kaggle/input/datasets/isabellouisedelgado/high-gothic-novels/WALPOLE_HORACE_CASTLE_OF_OTRANTO-pg696.txt',
  'WALPOLE HORACE CASTLE OF OTRANTO')]

In [7]:
LIB = pd.DataFrame(book_data, columns=['book_id','source_file_path', 'raw_title'])\
    .set_index('book_id').sort_index()
LIB

,source_file_path,raw_title
book_id,,
696,/kaggle/input/datasets/isabellouisedelgado/hig...,WALPOLE HORACE CASTLE OF OTRANTO
3268,/kaggle/input/datasets/isabellouisedelgado/hig...,RADCLIFFE ANN THE MYSTERIES OF UDOLPHO
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,REEVE CLARA THE OLD ENGLISH BARON


In [8]:
try:
    LIB['author'] = LIB.raw_title.apply(lambda x: ', '.join(x.split()[:2]))
    LIB['title'] = LIB.raw_title.apply(lambda x: ' '.join(x.split()[2:]))
    LIB = LIB.drop('raw_title', axis=1)

# Skip if this cell has already been run
except AttributeError:
    pass

LIB

,source_file_path,author,title
book_id,,,
696,/kaggle/input/datasets/isabellouisedelgado/hig...,"WALPOLE, HORACE",CASTLE OF OTRANTO
3268,/kaggle/input/datasets/isabellouisedelgado/hig...,"RADCLIFFE, ANN",THE MYSTERIES OF UDOLPHO
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,"REEVE, CLARA",THE OLD ENGLISH BARON


In [9]:
LIB['chap_pat'] = pd.Series({pat[0]:pat[1] for pat in chap_pats})
LIB

,source_file_path,author,title,chap_pat
book_id,,,,
696,/kaggle/input/datasets/isabellouisedelgado/hig...,"WALPOLE, HORACE",CASTLE OF OTRANTO,^CHAPTER\s+[IVXLCM]+\.$
3268,/kaggle/input/datasets/isabellouisedelgado/hig...,"RADCLIFFE, ANN",THE MYSTERIES OF UDOLPHO,^(?:VOLUME\s+\d+|\ CHAPTER\s+[IVXLCM]+)$
5182,/kaggle/input/datasets/isabellouisedelgado/hig...,"REEVE, CLARA",THE OLD ENGLISH BARON,^THE OLD ENGLISH BARON: A GOTHIC STORY\.$


In [10]:
# Add clip_pats and chap_pat to LIB
LIB['clip_pats'] = None
LIB['chap_pat'] = None

for i in LIB.index:
    LIB.at[i, 'clip_pats'] = clip_pats_gutenberg

for book_id, pat in chap_pats:
    LIB.at[book_id, 'chap_pat'] = pat

# Verify it looks right before running corpus
LIB[['title', 'clip_pats', 'chap_pat']]

,title,clip_pats,chap_pat
book_id,,,
696,CASTLE OF OTRANTO,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^CHAPTER\s+[IVXLCM]+\.$
3268,THE MYSTERIES OF UDOLPHO,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^(?:VOLUME\s+\d+|\ CHAPTER\s+[IVXLCM]+)$
5182,THE OLD ENGLISH BARON,"[\*\*\*\s*START OF, \*\*\*\s*END OF]",^THE OLD ENGLISH BARON: A GOTHIC STORY\.$


In [11]:
def tokenize_source(book_id):
    src_file = LIB.loc[book_id].source_file_path
    
    # Convert the raw text into a dataframe
    text_lines = open(src_file, 'r', encoding="utf-8-sig").readlines()
    LINES = pd.DataFrame({'book_id': book_id, 'line_str': text_lines})
    LINES.line_str = LINES.line_str.str.strip()
    
    # Clip the cruft

    clip_pats = LIB.loc[book_id, 'clip_pats']
    start_pat = clip_pats[0]
    end_pat = clip_pats[1]
    start = LINES.line_str.str.contains(start_pat, regex=True)
    end = LINES.line_str.str.contains(end_pat, regex=True)
    try:
        start_line_num = LINES.loc[start].index[0]
    except IndexError:
        raise ValueError("Clip start pattern not found.")
    try:
        end_line_num = LINES.loc[end].index[0]
    except IndexError:
        raise ValueError("Clip end pattern not found.")
    LINES = LINES.loc[start_line_num + 1 : end_line_num - 2].copy()
    
    # Chunk chapters
    chap_pat = LIB.loc[book_id, 'chap_pat']
    chap_lines = LINES.line_str.str.contains(chap_pat, regex=True, case=True)
    LINES.loc[chap_lines, 'chap_num'] = [i+1 for i in range(LINES.loc[chap_lines].shape[0])]
    LINES.chap_num = LINES.chap_num.ffill()
    LINES = LINES.loc[~LINES.chap_num.isna()]
    LINES = LINES.loc[~chap_lines]
    LINES.chap_num = LINES.chap_num.astype('int')
    CHAPS = LINES.groupby('chap_num', group_keys=True)['line_str']\
        .apply(lambda x: '\n'.join(x)).to_frame('chap_str')
    del LINES
    
    # Build OHCO records with plain Python loops
    records = []
    for chap_num, chap_str in CHAPS.chap_str.items():
        for para_num, para_str in enumerate(chap_str.split('\n\n')):
            if not para_str.strip():
                continue
            for sent_num, sent_str in enumerate(nltk.sent_tokenize(para_str)):
                for token_num, (token_str, pos) in enumerate(nltk.pos_tag(nltk.word_tokenize(sent_str))):
                    records.append((chap_num, para_num, sent_num, token_num, token_str, pos))
    del CHAPS
    
    # Build TOKENS dataframe
    TOKENS = pd.DataFrame(records, columns=['chap_num', 'para_num', 'sent_num', 'token_num', 'token_str', 'pos'])
    TOKENS['pos_group'] = TOKENS.pos.str[:2]
    TOKENS['term_str'] = TOKENS.token_str.str.lower().str.replace(r"[\W_]+", "", regex=True)
    TOKENS = TOKENS[TOKENS.term_str != ''].copy()
    TOKENS = TOKENS.set_index(['chap_num', 'para_num', 'sent_num', 'token_num'])
    
    return TOKENS

In [12]:
corpus = []
for i in LIB.index:
    print(LIB.loc[i].title)
    corpus.append(tokenize_source(i))
print("Done")
CORPUS = pd.concat(corpus, keys=LIB.index)
CORPUS.index.names = OHCO
del corpus

CASTLE OF OTRANTO
THE MYSTERIES OF UDOLPHO
THE OLD ENGLISH BARON
Done


In [13]:
CORPUS

token_str  pos pos_group  \
book_id chap_num para_num sent_num token_num                               
696     1        1        0        0              Manfred  NNP        NN   
                                   2               Prince  NNP        NN   
                                   3                   of   IN        IN   
                                   4              Otranto  NNP        NN   
                                   6                  had  VBD        VB   
...                                                   ...  ...       ...   
5182    1        1302     1        20                 and   CC        CC   
                                   21                 the   DT        DT   
                                   22           certainty   NN        NN   
                                   23                  of   IN        IN   
                                   24         RETRIBUTION  NNP        NN   

                                                 term_str  
book_id chap_num para_num sent_num token_num               
696     1        1        0        0              manfred  
                                   2               prince  
                                   3                   of  
                                   4              otranto  
                                   6                  had  
...                                                   ...  
5182    1        1302     1        20                 and  
                                   21                 the  
                                   22           certainty  
                                   23                  of  
                                   24         retribution  

[381711 rows x 4 columns]

In [14]:
CORPUS.sample(20)

token_str   pos pos_group  \
book_id chap_num para_num sent_num token_num                                
3268    4        194      1        7                 fire    NN        NN   
5182    1        1038     1        5               Oswald   NNP        NN   
                 237      0        26            attempts   NNS        NN   
696     5        64       1        2                  the    DT        DT   
3268    2        341      0        53                self    NN        NN   
        4        785      0        21         endeavoured   VBD        VB   
                 793      0        27                 had   VBD        VB   
                 788      0        23                real    JJ        JJ   
                 726      5        8                   is   VBZ        VB   
5182    1        307      0        29                that    DT        DT   
3268    2        323      2        30               their  PRP$        PR   
        3        212      2        32               given   VBN        VB   
5182    1        1076     1        44                 the    DT        DT   
3268    1        206      0        24            trembled   VBN        VB   
                 207      0        9                 hour    NN        NN   
        4        595      4        21               noble    JJ        JJ   
        1        602      3        48             herself    NN        NN   
        3        745      3        15                that   WDT        WD   
        1        380      2        13            interred   VBN        VB   
5182    1        780      8        31              assize    NN        NN   

                                                 term_str  
book_id chap_num para_num sent_num token_num               
3268    4        194      1        7                 fire  
5182    1        1038     1        5               oswald  
                 237      0        26            attempts  
696     5        64       1        2                  the  
3268    2        341      0        53                self  
        4        785      0        21         endeavoured  
                 793      0        27                 had  
                 788      0        23                real  
                 726      5        8                   is  
5182    1        307      0        29                that  
3268    2        323      2        30               their  
        3        212      2        32               given  
5182    1        1076     1        44                 the  
3268    1        206      0        24            trembled  
                 207      0        9                 hour  
        4        595      4        21               noble  
        1        602      3        48             herself  
        3        745      3        15                that  
        1        380      2        13            interred  
5182    1        780      8        31              assize

In [15]:
LIB.to_csv('/kaggle/working/LIB_high_gothic.csv')
CORPUS.to_csv('/kaggle/working/CORPUS_high_gothic.csv')